In [1]:
import numpy as np
import pandas as pd
import faiss
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.preprocessing import LabelEncoder,QuantileTransformer
import torch
import pickle
from collections import defaultdict
from sentence_transformers import SentenceTransformer
import os
import json
import ollama

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

from utils.features import feature_list
from utils.prompt_template import generate_prompt
from utils.alert_template import generate_alert

/home/abdulrahimfami/Desktop/final_poject/myenv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-23 11:28:52.365623: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [ ]:
FEATURES = feature_list
device = 'cuda' if torch.cuda.is_available() else 'cpu'

cuda


In [3]:
import gc

def clear_memory():

    print("Allocated:", torch.cuda.memory_allocated() / 1024**3, "GB")
    print("Reserved :", torch.cuda.memory_reserved() / 1024**3, "GB")
    print("Total GPU:", torch.cuda.get_device_properties(0).total_memory / 1024**3, "GB")

    gc.collect()
    torch.cuda.empty_cache()

clear_memory()

Allocated: 0.0 GB
Reserved : 0.0 GB
Total GPU: 3.62762451171875 GB


In [4]:
class Detector:
    '''
    This class is responsible for loading the trained TabNet model for attack detection, along with the necessary preprocessing tools such as the Quantile Transformer and Label Encoder. It provides functionality to perform predictions based on the loaded model and preprocessors, allowing the system to identify potential attacks based on input data.
    '''

    def __init__(self,ROOT =r"models/Detector"):
        
        self.model =  TabNetClassifier(device_name=device)
        self.model.load_model(os.path.join(ROOT,'tabnet_model.zip'))

        with open(os.path.join(ROOT,"quantile_transformer.pkl"), "rb") as f:
            self.qt = pickle.load(f)

        with open(os.path.join(ROOT,"label_encoder.pkl"), "rb") as f:
            self.le = pickle.load(f)
        
    
    def predict(self,sample):
        '''
        This function takes a single sample, preprocesses it, and uses the trained TabNet model to predict the attack class. It then generates a local explanation for the prediction, identifying the most influential features and their contribution percentages. Finally, it formats this information into a structured alert message that highlights the predicted threat category and the key contributing features.
        '''

        sample = sample.iloc[:1]

        X = sample[FEATURES]
        X_scaled = self.qt.transform(X).astype(np.float32)
        sample = sample.drop(columns=['id','Flow ID','Attempted Category','Label'],errors='ignore').to_dict(orient='records')

        
        explain_matrix,_ = self.model.explain(X_scaled)

        feature_scores = [(feat,score) for feat,score in zip(FEATURES,explain_matrix[0]) if score > 0]
        feature_scores.sort(key=lambda x: x[1], reverse=True)
        total = sum(value for _,value in feature_scores)
        percentage_scores = [(name,(val/total)*100) for name,val in feature_scores]


        top_features_text = "\n".join(
        [f"  • {name}: {pct:.5f}%" for name, pct in percentage_scores])

        alert_class = self.le.inverse_transform(self.model.predict(X_scaled))[0]
        alert = generate_alert(alert_class=alert_class,top_features_text=top_features_text)
        sample = json.dumps(sample[0])

        return sample,alert,alert_class


    def performance_evaluation(self,x_test,y_test):
        pass



In [5]:
class Retreiver:
    '''
    This class is designed to handle the retrieval of relevant techniques from the MITRE dataset based on vector search results. It utilizes the loaded FAISS index and metadata to find and return information about techniques that are similar to the input query, allowing analysts to quickly access relevant techniques for further investigation.
    '''
    
    def __init__(self):

        model_path,knowledge_base_path = r"models/Embedder",r"Knowledge_base"
    
        self.embedder = SentenceTransformer(os.path.join(model_path))
        self.vector_data = faiss.read_index(os.path.join(knowledge_base_path,r"mitre_index.faiss"))

        with open(os.path.join(knowledge_base_path, "mitre_metadata.json"), "r") as f:
            self.mitre_metadata = json.load(f)

        with open(os.path.join(knowledge_base_path, "mitre_lookup.json"), "r") as f:
            self.mitre_lookup = json.load(f)
        
        
        
    
    def vector_search(self,alert_query,K=10):

        '''
        This function takes an alert query, encodes it using the embedder, and performs a vector search against the MITRE dataset. It aggregates scores for each technique based on occurrence and similarity, and returns a ranked list of techniques with their combined scores and metadata.
        '''
        
        query_embed = self.embedder.encode([alert_query],normalize_embeddings=True,convert_to_numpy=True,device=device)
        D,I = self.vector_data.search(np.array(query_embed).astype('float32'),K)

        technique_scores = defaultdict(lambda:{'count':0,'score_sum':0,'metadata':None})

        for score,idx in zip(D[0],I[0]):
            
            entry = self.mitre_metadata[str(idx)]
            tech_id = entry['metadata']['id']

            technique_scores[tech_id]['count'] += 1
            technique_scores[tech_id]['score_sum']+= float(score)
            technique_scores[tech_id]['metadata'] = entry['metadata']
        
        final_results = []

        for tech_id,data in technique_scores.items():
            
            occurence_score,similarity_score = data['count'],data['score_sum']
            combined_score = (0.7*occurence_score) + (0.3*similarity_score)

            final_results.append((tech_id,combined_score,occurence_score,data['metadata']))
            final_results.sort(key=lambda x:x[1],reverse=True)

        mitre_tech_context = ""
        # Description: {json.dumps(self.mitre_lookup[tech_id])}

        for tech_id,score,occurence,metadata in final_results:
            mitre_tech_context += f"""
        Technique ID: {tech_id}
        Name: {metadata.get('name')}
        Score: {score:.2f}
        """
        return final_results,mitre_tech_context.strip()
        pass
    


    

In [6]:
class Generator:
    '''
    This class is responsible for generating reports and summaries based on the analysis performed by the Detector and Retriever classes. It formats the output in a user-friendly manner, making it easier for analysts to understand and act upon the findings.
    '''
    pass

    def __init__(self):
        self.model = 'mistral:7b'
    

    def construct_prompt():
        '''
        This function constructs a prompt for generating a report based on the alert and the retrieved MITRE techniques. It combines the alert information with the context of the relevant techniques to create a comprehensive prompt that can be used to generate a detailed report or summary.
        '''
        pass


    def generate_report(self,sample,alert,mitre_context,prompt=None):
        '''
        This function takes the constructed prompt and uses a language model to generate a report or summary. It processes the prompt and produces a coherent and informative output that summarizes the findings from the alert and the retrieved techniques, providing actionable insights for analysts.
        '''

        if not prompt:
            prompt = generate_prompt(alert=alert,mitre_context=mitre_context,sample=sample)

        stream = ollama.generate(
            model=self.model,
            prompt=prompt,
            stream=True,
        )

        for chunk in stream:
            print(chunk['response'],end='',flush=True)
        
        return stream

    

In [7]:
class SOC_ANALYST:
    pass

    def __init__(self):
        self.detector = Detector()
        self.retriever = Retreiver()
        self.generator = Generator()

In [8]:
detector = Detector()
retreiver = Retreiver()
generator = Generator()

/home/abdulrahimfami/Desktop/final_poject/myenv/lib/python3.10/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


# Testing Samples

## 1. Botnet Attack  [T1583/T1584]

In [9]:
attack_data = pd.read_csv(r"attack/Botnet/Botnet_1.csv")

sample,alert,alert_class = detector.predict(sample=attack_data)
print(alert)

🚨 SECURITY ALERT: Suspicious Activity Detected

⚠️ Predicted Threat Category: Botnet

The system has identified unusual network behavior
consistent with the attack class: Botnet.

🔎 Contributing Features (Local Explanation):
  • PSH Flag Count: 27.14991%
  • Fwd Seg Size Min: 25.20379%
  • FWD Init Win Bytes: 18.33236%
  • Bwd IAT Mean: 18.24701%
  • Subflow Fwd Bytes: 7.84819%
  • Subflow Bwd Bytes: 3.21874%

-------------------------------------------------
This explanation represents the model’s local
feature attribution for the current sample.


In [12]:
technique_info,mitre_context = retreiver.vector_search(alert_query=alert_class)

In [13]:
report = generator.generate_report(alert=alert_class,mitre_context=mitre_context,sample=sample)

 Primary MITRE Technique:
Technique ID: T1584 > T1584.005
Technique Name: Compromise Infrastructure > Command and Control (C2) Beaconing
Reason: The predicted attack category is Botnet, and the detected traffic pattern matches the T1584 technique for compromising infrastructure. The beaconing sub-technique (T1584.005) is relevant due to the frequent communication observed between the source IP and the destination IP.

Incident Overview:
A botnet attack has been detected based on the network flow data. The attacker establishes a connection with a command and control server, potentially allowing them to execute malicious commands or download additional payloads for further exploitation.

MITRE Technique Mapping:
* Technique ID: T1584 > T1584.005
* Technique Name: Compromise Infrastructure > Command and Control (C2) Beaconing
* Short explanation of why this technique matches the attack: The source IP initiates frequent communication with a destination IP using a specific port, which is co

## 2. PortScan Attack  [T1046]

In [16]:
attack_data = pd.read_csv(r"attack/Portscan/Portscan_1.csv")

sample,alert,alert_class = detector.predict(sample=attack_data)
print(alert)

🚨 SECURITY ALERT: Suspicious Activity Detected

⚠️ Predicted Threat Category: Portscan

The system has identified unusual network behavior
consistent with the attack class: Portscan.

🔎 Contributing Features (Local Explanation):
  • Fwd PSH Flags: 23.53904%
  • FWD Init Win Bytes: 21.59698%
  • Fwd Act Data Pkts: 20.38523%
  • FIN Flag Count: 19.39909%
  • Flow IAT Std: 15.07967%

-------------------------------------------------
This explanation represents the model’s local
feature attribution for the current sample.


In [18]:
technique_info,mitre_context = retreiver.vector_search(alert_query=alert_class)

In [19]:
report = generator.generate_report(alert=alert_class,mitre_context=mitre_context,sample=sample)


 Primary MITRE Technique:
Technique ID: T1046 > T1046.001
Technique Name: Network Service Scanning > Port Scanning
Reason: The detected network flow exhibits a pattern consistent with a port scan, as indicated by the Predicted Attack Category and the high score for the technique 'T1046'.

Incident Overview:
A port scan attack has been detected on the network. This type of attack involves an unauthorized attempt to gain information about the open ports on a targeted system. The significance of this incident is that it potentially exposes vulnerabilities that could be exploited by the attacker, leading to further attacks or data breaches.

MITRE Technique Mapping:
* Technique ID: T1046 > T1046.001
* Technique Name: Network Service Scanning > Port Scanning
* Short explanation of why this technique matches the attack: The network flow shows a single packet from the source IP to the destination IP on an unusual port (33264 and 41511), which is indicative of a port scan.

Network Attribution

## 3. DoS Attack  [T1498/T1499]

In [20]:
attack_data = pd.read_csv(r"attack/DoS/DoS_1.csv")

sample,alert,alert_class = detector.predict(sample=attack_data)
print(alert)

🚨 SECURITY ALERT: Suspicious Activity Detected

⚠️ Predicted Threat Category: DoS

The system has identified unusual network behavior
consistent with the attack class: DoS.

🔎 Contributing Features (Local Explanation):
  • Bwd IAT Mean: 76.30193%
  • Packet Length Min: 11.86014%
  • Subflow Fwd Bytes: 7.95910%
  • Bwd IAT Std: 3.59764%
  • Total TCP Flow Time: 0.28118%

-------------------------------------------------
This explanation represents the model’s local
feature attribution for the current sample.


In [21]:
technique_info,mitre_context = retreiver.vector_search(alert_query=alert_class)

In [22]:
report = generator.generate_report(alert=alert_class,mitre_context=mitre_context,sample=sample)


 Primary MITRE Technique:
Technique ID: T1498.001
Technique Name: Direct Network Flood > Direct Denial of Service (DoS)
Reason: The predicted attack category is a DoS attack, and the observed traffic exhibits characteristics consistent with a direct network flood, such as a high number of packets from the source IP to the destination IP over an extended period (Flow Duration).

Incident Overview:
The detected attack involves a potential Denial-of-Service (DoS) attempt targeting the SSH service on the destination IP. The large number of incoming packets to port 22 from the source IP over a prolonged duration could consume network resources, potentially causing a disruption in services.

MITRE Technique Mapping:
* Technique ID: T1498.001
* Technique Name: Direct Network Flood > Direct Denial of Service (DoS)
* Short explanation of why this technique matches the attack: The observed traffic characteristics, such as high packet count and flow duration, correspond to a direct network flood 

## 4. SQL Injection

In [44]:
attack_data = pd.read_csv(r"attack/SQL Injection/SQL Injection_1.csv")


sample,alert,alert_class = detector.predict(sample=attack_data)
print(alert)

🚨 SECURITY ALERT: Suspicious Activity Detected

⚠️ Predicted Threat Category: SQL Injection

The system has identified unusual network behavior
consistent with the attack class: SQL Injection.

🔎 Contributing Features (Local Explanation):
  • Fwd Segment Size Avg: 22.08611%
  • Bwd IAT Max: 17.34286%
  • FWD Init Win Bytes: 16.80198%
  • Subflow Fwd Bytes: 11.87929%
  • Bwd Header Length: 9.36069%
  • Fwd Act Data Pkts: 8.67687%
  • Bwd Packet Length Std: 6.95898%
  • Fwd Packet Length Min: 6.89322%

-------------------------------------------------
This explanation represents the model’s local
feature attribution for the current sample.


In [49]:
technique_info,mitre_context = retreiver.vector_search(alert_query=alert_class)


In [ ]:
report = generator.generate_report(alert=alert_class,mitre_context=mitre_context,sample=sample)

 Primary MITRE Technique:
Technique ID: T1190.001
Technique Name: Exploitation of Remote Services (Web-based)
Reason: The network behavior exhibits characteristics consistent with SQL Injection, which falls under the category of Exploit Public-Facing Application in the MITRE ATT&CK framework.

Incident Overview:
A potential SQL injection attack has been identified based on the unusual network behavior observed. SQL injection is a code injection technique that attacks data-driven applications with the intent to alter the functionality of a database or application, extract sensitive data, or cause damage. This type of attack can lead to significant security breaches and unauthorized access to sensitive information.

MITRE Technique Mapping:
* Technique ID: T1190.001 (Exploitation of Remote Services (Web-based))
* Technique Name: Exploit Public-Facing Application
* Short explanation: The detected attack leverages a vulnerability in a web application to execute SQL injection, which falls u

## 5. XSS Attack [T1059,T1189]

In [57]:
attack_data = pd.read_csv(r"attack/XSS Attack/XSS Attack_1.csv")


sample,alert,alert_class = detector.predict(sample=attack_data)
print(alert)

🚨 SECURITY ALERT: Suspicious Activity Detected

⚠️ Predicted Threat Category: XSS Attack

The system has identified unusual network behavior
consistent with the attack class: XSS Attack.

🔎 Contributing Features (Local Explanation):
  • Subflow Fwd Bytes: 37.40715%
  • FWD Init Win Bytes: 20.23180%
  • Total TCP Flow Time: 19.57879%
  • Packet Length Min: 17.38838%
  • Total Bwd packets: 4.04479%
  • Fwd Packets/s: 0.77200%
  • Bwd Header Length: 0.57710%

-------------------------------------------------
This explanation represents the model’s local
feature attribution for the current sample.


In [59]:
technique_info,mitre_context = retreiver.vector_search(alert_query=alert_class)
print(mitre_context)

Technique ID: T1027.006
        Name: HTML Smuggling
        Score: 3.42
        
        Technique ID: T1059.007
        Name: JavaScript
        Score: 1.71
        
        Technique ID: T1555.003
        Name: Credentials from Web Browsers
        Score: 0.86
        
        Technique ID: T1059.005
        Name: Visual Basic
        Score: 0.85
        
        Technique ID: T1189
        Name: Drive-by Compromise
        Score: 0.85
        
        Technique ID: T1185
        Name: Browser Session Hijacking
        Score: 0.85


In [60]:
report = generator.generate_report(alert=alert_class,mitre_context=mitre_context,sample=sample)

 Primary MITRE Technique:
Technique ID: T1059 > T1059.007
Technique Name: Command and Scripting Interpreter: JavaScript
Reason: The predicted attack category is XSS (Cross-Site Scripting) Attack, which often involves injecting malicious scripts into web pages to execute arbitrary code in users' browsers. In this incident, the network flow record shows a connection between the source IP and destination port 9009, which is commonly used for Node.js applications, suggesting JavaScript execution may be involved.

Incident Overview:
A potential Cross-Site Scripting (XSS) attack has been detected on our network. XSS attacks can allow an attacker to inject malicious scripts into web pages viewed by other users, leading to data theft, unauthorized account takeover, and further network compromise. In this incident, the source IP initiated a connection to port 9009, which is typically associated with Node.js applications, indicating a possible JavaScript execution related to XSS attack.

MITRE T

## 6. BruteForce [T1110]

In [61]:
attack_data = pd.read_csv(r"attack/BruteForce/BruteForce_1.csv")


sample,alert,alert_class = detector.predict(sample=attack_data)
print(alert)

🚨 SECURITY ALERT: Suspicious Activity Detected

⚠️ Predicted Threat Category: BruteForce

The system has identified unusual network behavior
consistent with the attack class: BruteForce.

🔎 Contributing Features (Local Explanation):
  • Bwd Header Length: 26.21211%
  • FIN Flag Count: 21.87384%
  • Fwd Packets/s: 16.17147%
  • Average Packet Size: 13.45705%
  • Bwd IAT Mean: 10.82761%
  • Bwd Segment Size Avg: 7.54276%
  • Fwd PSH Flags: 2.30161%
  • Bwd IAT Std: 1.61354%

-------------------------------------------------
This explanation represents the model’s local
feature attribution for the current sample.


In [ ]:
technique_info,mitre_context = retreiver.vector_search(alert_query=alert_class)


In [68]:
report = generator.generate_report(alert=alert_class,mitre_context=mitre_context,sample=sample)

 Primary MITRE Technique:
Technique ID: T1110 > T1110.001
Technique Name: Brute Force > Password Guessing
Reason: The network flow record shows multiple attempts (Total Fwd Packets and Total Bwd Packets both equal 6) over an extended duration (Flow Duration equals 4008190) from a single source IP to the same destination IP, which aligns with the behavior expected during password guessing attacks.

Incident Overview:
A brute force attack targeting password guessing was detected on the network. The attacker is attempting multiple login attempts using automated tools, increasing the risk of unauthorized access and potential data breaches.

MITRE Technique Mapping:
* Technique ID: T1110 > T1110.001
* Technique Name: Brute Force > Password Guessing
* Short explanation of why this technique matches the attack: The observed traffic characteristics, such as multiple attempts and prolonged duration, suggest that an attacker is attempting to crack passwords through brute force methods.

Network 

## 7. Infilteration

In [69]:
attack_data = pd.read_csv(r"attack/Infiltration/Infiltration_2.csv")


sample,alert,alert_class = detector.predict(sample=attack_data)
print(alert)

technique_info,mitre_context = retreiver.vector_search(alert_query=alert_class)

report = generator.generate_report(alert=alert_class,mitre_context=mitre_context,sample=sample)

🚨 SECURITY ALERT: Suspicious Activity Detected

⚠️ Predicted Threat Category: Infiltration

The system has identified unusual network behavior
consistent with the attack class: Infiltration.

🔎 Contributing Features (Local Explanation):
  • Total Bwd packets: 36.98262%
  • FWD Init Win Bytes: 34.72126%
  • Subflow Fwd Bytes: 12.94004%
  • Average Packet Size: 8.10237%
  • Bwd Header Length: 4.76234%
  • Subflow Bwd Bytes: 2.49137%

-------------------------------------------------
This explanation represents the model’s local
feature attribution for the current sample.
Technique ID: T1052
        Name: Exfiltration Over Physical Medium
        Score: 0.89
        
        Technique ID: T1048
        Name: Exfiltration Over Alternative Protocol
        Score: 0.87
        
        Technique ID: T1020
        Name: Automated Exfiltration
        Score: 0.87
        
        Technique ID: T1029
        Name: Scheduled Transfer
        Score: 0.87
        
        Technique ID: T1011
     